In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

# Task 1: Write your code here:
# Load the dataset
df_path = os.path.join(path, 'Q3_data.csv')

df = pd.read_csv(df_path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
# first 5 rows
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)
#we can see the Missing_Percentage for some column is 99%!
#I don't see droping na is the right solotion here
#I will drop the entire column because too many values are missing
df= df.drop(columns=['D_87','D_88','B_39','D_110','D_111','D_108','B_42','D_73','D_135','D_136'])


In [ ]:
# Task 2: Write your code here:
#Do we have duplicate samples?

def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# No need because from info above we can see thers is no Object type


In [ ]:
from sklearn.preprocessing import StandardScaler

# Task 4: Write your code here:
features = df.columns.drop("Target")  # WE DON'T SCALE THE TARGET
numerical_cols = df[features].select_dtypes(include='number').columns
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Task 5: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")
#we can see the our target is imbalance so we must use Stratification
#to forces the Train and Test splits to preserve the same class ratios as the original dataset.

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target']

In [ ]:
%pip install catboost
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier


In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for logistic regression results for each fold
lr_accuracy = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    model = CatBoostClassifier(verbose=0,n_estimators=200,max_depth=4)

    # Train the model
    model.fit(X_train, y_train)
    # Predict on the test set
    y_pred = model.predict(X_test)


    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_f1.append(f1)
print(' averaged accuracy:',np.mean(np.array(lr_accuracy)))
print(' averaged f1: ',np.mean(np.array(lr_accuracy)))

In [ ]:
# Task 1: Write your code here:


importance = list(zip(X.columns, model.feature_importances_))
sorted_importance = sorted(importance, key=lambda x: abs(x[1]), reverse=True)

# Extract sorted features and their coefficients
features, coefficients = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(20, 15))
plt.barh(features, coefficients, color='darkblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:
print("golden feature: P_2")

In [ ]:
# Task Bonus: Write your code here: